# TTA Example

## Imports and Configs

In [ ]:
import sys
from os import path, environ
from argparse import ArgumentParser

import torch
from torchinfo import summary

from ttadapters import datasets, models, methods
from ttadapters.utils import visualizer, validator
from ttadapters.datasets import DatasetHolder, scenarios

In [ ]:
environ["TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS"] = "1"
environ["TORCHDYNAMO_CAPTURE_DYNAMIC_OUTPUT_SHAPE_OPS"] = "1"

torch._dynamo.config.capture_scalar_outputs = True
torch._dynamo.config.suppress_errors = True

In [ ]:
environ["CUDA_VISIBLE_DEVICES"] = "0"

### Parse Arguments

In [ ]:
# Set Batch Size
BATCH_SIZE = 2, 8, 1  # Local
#BATCH_SIZE = 40, 200, 1  # A100 or H100
ACCUMULATE_STEPS = 1

# Set Data Root
DATA_ROOT = path.join(".", "data")

# Set Target Dataset
SOURCE_DOMAIN = datasets.SHIFTDataset

# Set Model List
MODEL_ZOO = ["rcnn", "swinrcnn", "yolo11", "rtdetr"]
MODEL_TYPE = MODEL_ZOO[0]

In [ ]:
# Create argument parser
parser = ArgumentParser(description="Adaptation experiment script for Test-Time Adapters")

# Add model arguments
parser.add_argument("--dataset", type=str, choices=["shift", "city"], default="shift", help="Training dataset")
parser.add_argument("--model", type=str, choices=MODEL_ZOO, default=MODEL_TYPE, help="Model architecture")

# Add training arguments
parser.add_argument("--train-batch", type=int, default=BATCH_SIZE[0], help="Training batch size")
parser.add_argument("--valid-batch", type=int, default=BATCH_SIZE[1], help="Validation batch size")
parser.add_argument("--accum-step", type=int, default=ACCUMULATE_STEPS, help="Gradient accumulation steps")
parser.add_argument("--data-root", type=str, default=DATA_ROOT, help="Root directory for datasets")
parser.add_argument("--device", type=int, default=0, help="CUDA device number")
parser.add_argument("--additional_gpu", type=int, default=0, help="Additional CUDA device count")
parser.add_argument("--use-bf16", action="store_true", help="Use bfloat16 precision")

# Parsing arguments
if "ipykernel" in sys.modules:
    args = parser.parse_args([])
    print("INFO: Running in notebook mode with default arguments")
else:
    args = parser.parse_args()

# Update global variables based on parsed arguments
BATCH_SIZE = args.train_batch, args.valid_batch, BATCH_SIZE[2]
ACCUMULATE_STEPS = args.accum_step
DATA_ROOT = args.data_root
MODEL_TYPE = args.model
match args.dataset:
    case "shift":
        SOURCE_DOMAIN = datasets.SHIFTDataset
    case "city":
        SOURCE_DOMAIN = datasets.CityScapesDataset
    case _:
        raise ValueError(f"Unsupported dataset: {args.dataset}")
print(f"INFO: Set batch size - Train: {BATCH_SIZE[0]}, Valid: {BATCH_SIZE[1]}, Test: {BATCH_SIZE[2]}")

### Check GPU Availability

In [ ]:
!nvidia-smi

In [ ]:
# Set CUDA Device Number
DEVICE_NUM = 0 if not args.device else args.device
ADDITIONAL_GPU = 0 if not args.additional_gpu else args.additional_gpu
DATA_TYPE = torch.float32 if not args.use_bf16 else torch.bfloat16

if torch.cuda.is_available():
    if ADDITIONAL_GPU:
        torch.cuda.set_device(DEVICE_NUM)
        device = torch.device("cuda")
    else:
        device = torch.device(f"cuda:{DEVICE_NUM}")
else:
    device = torch.device("cpu")
    DEVICE_NUM = -1

print(f"INFO: Using device - {device}" + (f":{DEVICE_NUM}" if ADDITIONAL_GPU else ""))
print(f"INFO: Using data precision - {DATA_TYPE}")

## Define Dataset

In [ ]:
# Fast download patch
datasets.patch_fast_download_for_object_detection()

In [ ]:
# Basic pre-training dataset
match SOURCE_DOMAIN:
    case datasets.SHIFTDataset:
        # discrete
        dataset = DatasetHolder(
            train=datasets.SHIFTClearDatasetForObjectDetection(root=DATA_ROOT, train=True),
            valid=datasets.SHIFTClearDatasetForObjectDetection(root=DATA_ROOT, valid=True),
            test=datasets.SHIFTCorruptedDatasetForObjectDetection(root=DATA_ROOT, valid=True)
        )
        # continuous
        _ = datasets.SHIFTContinuous100DatasetForObjectDetection(root=DATA_ROOT)  # 100
        _ = datasets.SHIFTContinuous10DatasetForObjectDetection(root=DATA_ROOT)  # 10
        _ = datasets.SHIFTContinuousSubsetForObjectDetection(root=DATA_ROOT)  # 1 + split
    case datasets.CityScapesDataset:
        dataset = DatasetHolder(
            train=datasets.CityScapesDatasetForObjectDetection(root=DATA_ROOT, train=True),
            valid=datasets.CityScapesDatasetForObjectDetection(root=DATA_ROOT, valid=True),
            test=datasets.CityScapesCorruptedDatasetForObjectDetection(root=DATA_ROOT, valid=True)
        )
    case _:
        raise ValueError(f"Unsupported dataset: {SOURCE_DOMAIN}")

# Dataset info
CLASSES = dataset.test.classes
NUM_CLASSES = len(CLASSES)
print(f"INFO: Number of classes - {NUM_CLASSES} {CLASSES}")

In [ ]:
# Check annotation keys-values
dataset.test[999]

In [ ]:
# Check data shape
dataset.test[999][0].shape  # should be (num_channels, height, width)

## Load Base Model

In [ ]:
# Initialize base_model
match MODEL_TYPE:
    case "rcnn":
        base_model = models.FasterRCNNForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR_NATUREYOO if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case "swinrcnn":
        base_model = models.SwinRCNNForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR_NATUREYOO if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case "yolo11":
        DATA_TYPE = torch.bfloat16  # bf16 default
        base_model = models.YOLO11ForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case "rtdetr":
        DATA_TYPE = torch.bfloat16  # bf16 default
        base_model = models.RTDetrForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case _:
        raise ValueError(f"Unsupported model type: {MODEL_TYPE}")

print("INFO: Model state loaded -", load_result)
base_model.to(device)

In [ ]:
summary(base_model)

## Load Adaptation Method

In [ ]:
adaptive_model = base_model

## Evaluation

In [ ]:
# Load Pretrained APT Weights & Un-Freeze Model Encoder
# Allow FPN/Encoder to adapt during online adaptation
base_model.eval()
#adaptive_model.online()
summary(adaptive_model)

### Load Scenarios

In [ ]:
# Ensure split (required due to Scenario class works with coroutines)
_ = datasets.SHIFTContinuousSubsetForObjectDetection(root=DATA_ROOT, train=True)

In [ ]:
data_preparation = base_model.DataPreparation(datasets.base.BaseDataset(), evaluation_mode=True)

In [ ]:
import cv2
import numpy as np
from torchvision import tv_tensors

In [ ]:
def heq_adaptation(image: tv_tensors.Image) -> tv_tensors.Image:
    # TV ImageTensor => numpy
    image_np = image.permute(1, 2, 0).cpu().numpy()  # (H, W, C), RGB

    # float32 [0, 255] => uint8 [0, 255]
    image_np = image_np.astype(np.uint8)

    # Convert to YCrCb color space to preserve color information
    image_ycrcb = cv2.cvtColor(image_np, cv2.COLOR_RGB2YCrCb)

    # Apply histogram equalization only on Y channel (luminance)
    image_ycrcb[:, :, 0] = cv2.equalizeHist(image_ycrcb[:, :, 0])

    # Convert back to RGB
    image_rgb = cv2.cvtColor(image_ycrcb, cv2.COLOR_YCrCb2RGB)

    # uint8 => float32 [0, 255]
    image_rgb = image_rgb.astype(np.float32)

    # numpy to TV ImageTensor
    image_tensor = torch.from_numpy(image_rgb).permute(2, 0, 1)  # (C, H, W)
    image_tensor = tv_tensors.Image(image_tensor)

    return image_tensor

In [ ]:
sample = dataset.test[999][0]

In [ ]:
sample

In [ ]:
heq_adaptation(sample)

In [ ]:
def clahe_adaptation(image: tv_tensors.Image, clip_limit: float = 2.0, tile_size: int = 8) -> tv_tensors.Image:
    # TV ImageTensor => numpy
    image_np = image.permute(1, 2, 0).cpu().numpy()  # (H, W, C), RGB

    # float32 [0, 255] => uint8 [0, 255]
    image_np = image_np.astype(np.uint8)

    # Convert to YCrCb color space
    image_ycrcb = cv2.cvtColor(image_np, cv2.COLOR_RGB2YCrCb)

    # Apply histogram equalization on Y channel
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_size, tile_size))
    image_ycrcb[:, :, 0] = clahe.apply(image_ycrcb[:, :, 0])

    # Convert back to RGB
    image_rgb = cv2.cvtColor(image_ycrcb, cv2.COLOR_YCrCb2RGB)

    # uint8 => float32 [0, 255]
    image_rgb = image_rgb.astype(np.float32)

    # numpy to TV ImageTensor
    image_tensor = torch.from_numpy(image_rgb).permute(2, 0, 1)  # (C, H, W)
    image_tensor = tv_tensors.Image(image_tensor)

    return image_tensor

In [ ]:
def get_dark_channel(image, patch_size=15):
    """이미지의 dark channel을 계산"""
    min_channel = np.min(image, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (patch_size, patch_size))
    dark_channel = cv2.erode(min_channel, kernel)
    return dark_channel

def estimate_atmospheric_light(image, dark_channel, top_percent=0.001):
    """대기광(atmospheric light) 추정"""
    h, w = dark_channel.shape
    num_pixels = h * w
    num_brightest = int(max(num_pixels * top_percent, 1))
    
    dark_vec = dark_channel.reshape(num_pixels)
    image_vec = image.reshape(num_pixels, 3)
    
    indices = np.argsort(dark_vec)[::-1][:num_brightest]
    brightest_pixels = image_vec[indices]
    atmospheric_light = np.max(brightest_pixels, axis=0)
    
    return atmospheric_light

def get_transmission(image, atmospheric_light, omega=0.95, patch_size=15):
    """전달률(transmission) 추정"""
    normalized = image / atmospheric_light
    transmission = 1 - omega * get_dark_channel(normalized, patch_size)
    return transmission

def dehaze_adaptation(image: tv_tensors.Image, patch_size: int = 15, omega: float = 0.95, t0: float = 0.1, top_percent: float = 0.001) -> tv_tensors.Image:
    # TV ImageTensor => numpy
    image_np = image.permute(1, 2, 0).cpu().numpy()  # (H, W, C), RGB

    # float32 [0, 255] => uint8 [0, 255]
    image_np = image_np.astype(np.uint8)

    # Dark Channel Prior 알고리즘 적용
    image_float = image_np.astype(np.float64) / 255.0
    
    dark_channel = get_dark_channel(image_float, patch_size)
    atmospheric_light = estimate_atmospheric_light(image_float, dark_channel, top_percent)
    transmission = get_transmission(image_float, atmospheric_light, omega, patch_size)
    transmission = np.maximum(transmission, t0)
    
    result = np.zeros_like(image_float)
    for i in range(3):
        result[:, :, i] = (image_float[:, :, i] - atmospheric_light[i]) / transmission + atmospheric_light[i]
    
    result = np.clip(result * 255, 0, 255).astype(np.uint8)

    # uint8 => float32 [0, 255]
    result = result.astype(np.float32)

    # numpy to TV ImageTensor
    image_tensor = torch.from_numpy(result).permute(2, 0, 1)  # (C, H, W)
    image_tensor = tv_tensors.Image(image_tensor)

    return image_tensor

In [ ]:
def unsharpmask_adaptation(image: tv_tensors.Image, radius: float = 5.0, amount: float = 1.5, threshold: float = 0) -> tv_tensors.Image:
    # TV ImageTensor => numpy
    image_np = image.permute(1, 2, 0).cpu().numpy()  # (H, W, C), RGB
    
    # float32 [0, 255] => uint8 [0, 255]
    image_np = image_np.astype(np.uint8)
    
    # 가우시안 블러 적용
    # ksize는 (0, 0)으로 하면 sigma 값으로부터 자동 계산
    blurred = cv2.GaussianBlur(image_np, (0, 0), radius)
    
    # Unsharp mask 계산
    mask = cv2.subtract(image_np, blurred)
    
    # Threshold 적용 (선택적)
    if threshold > 0:
        # 절대값이 threshold 이상인 경우만 적용
        _, mask = cv2.threshold(np.abs(mask), threshold, 255, cv2.THRESH_TOZERO)
    
    # 선명화된 이미지 = 원본 + amount × mask
    sharpened = cv2.addWeighted(image_np, 1.0, mask, amount, 0)
    
    # 값 범위 클리핑
    sharpened = np.clip(sharpened, 0, 255)
    
    # uint8 => float32 [0, 255]
    sharpened = sharpened.astype(np.float32)
    
    # numpy to TV ImageTensor
    image_tensor = torch.from_numpy(sharpened).permute(2, 0, 1)  # (C, H, W)
    image_tensor = tv_tensors.Image(image_tensor)
    
    return image_tensor

In [ ]:
def combined_adaptation(image: tv_tensors.Image) -> tv_tensors.Image:
    image = clahe_adaptation(image)
    #image = dehaze_adaptation(image)
    #image = dehaze_adaptation(image, omega=0.80, t0=0.2, patch_size=7)
    #image = unsharpmask_adaptation(image)
    image = unsharpmask_adaptation(image, radius=2.0, amount=0.5, threshold=5)
    return image

In [ ]:
data_preparation = base_model.DataPreparation(datasets.base.BaseDataset(), evaluation_mode=True)

match SOURCE_DOMAIN:
    case datasets.SHIFTDataset:
        continual_scenario = scenarios.SHIFTDiscreteScenarioForContinualTTA(
            root=DATA_ROOT, valid=True, transforms=data_preparation.transforms,
            order=scenarios.SHIFTDiscreteScenarioForContinualTTA.WHWPAPER
        )
        gradual_scenario = scenarios.SHIFTContinuousScenarioForGradualTTA(
            root=DATA_ROOT, valid=True, transforms=data_preparation.transforms,
            order=scenarios.SHIFTContinuousScenarioForGradualTTA.DEFAULT
        )
    case datasets.CityScapesDataset:
        continual_scenario = None
        gradual_scenario = None
    case _:
        raise ValueError(f"Unsupported dataset: {SOURCE_DOMAIN}")

In [ ]:
methods = {
    #'Direct-Test': base_model,
    adaptive_model.model_name: adaptive_model
}

In [ ]:
evaluator = validator.DetectionEvaluator(list(methods.values()), classes=CLASSES, data_preparation=data_preparation, dtype=DATA_TYPE, device=device, no_grad=False)
evaluator_loader_params = dict(batch_size=BATCH_SIZE[2], shuffle=False, collate_fn=data_preparation.collate_fn)

In [ ]:
visualizer.visualize_metrics(continual_scenario(**evaluator_loader_params).play(evaluator, index=methods.keys()))

In [ ]:
#visualizer.visualize_metrics(gradual_scenario(**evaluator_loader_params).play(evaluator, index=methods.keys()))